# CEG-WM prospective Content texture stratification v1
User-only exploratory analysis. This notebook does not change any method, Gate, or scientific status. Run once; do not retry or resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
TARGET_BRANCH = 'stage-a-content-texture-stratification-v1'
SOURCE = Path('/content/cegwm-content-texture-stratification-v1-source')
LOCAL = Path('/content/cegwm-content-texture-stratification-v1-local')
SINK = Path('/content/drive/MyDrive/CEG-WM/content_texture_stratification_v1')
PROVENANCE = Path('/content/drive/MyDrive/CEG-WM')
if SOURCE.exists() or LOCAL.exists(): raise RuntimeError('initial-only local path exists')


In [ ]:
subprocess.run(['git','clone','--no-single-branch','--branch',TARGET_BRANCH,REPO_URL,str(SOURCE)], check=True, stdout=subprocess.DEVNULL)
EXPECTED_EXACT = subprocess.run(['git','rev-parse','HEAD'], cwd=SOURCE, check=True, capture_output=True, text=True).stdout.strip()
if subprocess.run(['git','branch','--show-current'], cwd=SOURCE, check=True, capture_output=True, text=True).stdout.strip() != TARGET_BRANCH: raise RuntimeError('branch differs')
if len(EXPECTED_EXACT) != 40 or subprocess.run(['git','status','--porcelain'], cwd=SOURCE, check=True, capture_output=True, text=True).stdout: raise RuntimeError('exact or clean state differs')
subprocess.run([os.sys.executable,'-m','pip','install','-e',str(SOURCE)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if subprocess.run(['git','rev-parse','HEAD'], cwd=SOURCE, check=True, capture_output=True, text=True).stdout.strip() != EXPECTED_EXACT or subprocess.run(['git','status','--porcelain'], cwd=SOURCE, check=True, capture_output=True, text=True).stdout: raise RuntimeError('post-install identity differs')


In [ ]:
from google.colab import userdata
import json
_root = userdata.get('CEG_WM_ROOT_KEY')
_token = userdata.get('HF_TOKEN')
_env = dict(os.environ); _env['CEG_WM_ROOT_KEY'] = _root; _env['HF_TOKEN'] = _token
_root = _token = None
_cmd = [os.sys.executable,'-m','experiments.run_content_texture_stratification_v1','--repo-root',str(SOURCE),'--expected-exact',EXPECTED_EXACT,'--local-work-root',str(LOCAL),'--artifact-sink',str(SINK),'--provenance-root',str(PROVENANCE)]
_accepted = None
try:
    _proc = subprocess.Popen(_cmd, cwd=SOURCE, env=_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    _env.pop('CEG_WM_ROOT_KEY', None); _env.pop('HF_TOKEN', None); _env = None
    for _line in _proc.stdout:
        if len(_line) > 4096: _proc.kill(); break
        if _line.startswith('CEGWM_TEXTURE_RESULT '):
            _value = json.loads(_line.split(' ',1)[1]); _accepted = _line.strip() if _accepted is None else None
    _rc = _proc.wait()
except Exception as _error:
    _env = None; _rc = 2; _accepted = None
if _rc not in (0,2) or _accepted is None:
    print('CEGWM_TEXTURE_HANDOFF_FAILURE '+json.dumps({'stage':'runner','error_class':'OperationalFailure'},separators=(',',':')))
else:
    print(_accepted)


In [ ]:
if _accepted is not None:
    _receipt = json.loads(_accepted.split(' ',1)[1])
    _run = SINK / EXPECTED_EXACT / _receipt['run_id'] / 'terminal'
    _zip = _run / (_receipt['run_id'] + '.zip'); _sha = _run / (_zip.name + '.sha256')
    _binding = _sha.read_text(encoding='ascii')
    if _binding != _receipt['terminal_sha256'] + '  ' + _zip.name + '\n' or not _zip.is_file(): raise RuntimeError('Drive terminal pair differs')
    print('CEGWM_TEXTURE_ARTIFACT '+json.dumps({'status':'artifact_pair_saved','exact':EXPECTED_EXACT,'run_id':_receipt['run_id'],'terminal_sha256':_receipt['terminal_sha256'],'drive_directory':str(_run)},sort_keys=True,separators=(',',':')))
